# Laboratorio: Algoritmos Genéticos - Curso 2
### Universidad de Cundinamarca - Seccional Ubaté
**Integrantes:** (Edwar Arley Fontecha Chiquiza y Daniel Esteban Chavez Marroquin)

## Ejercicios 1 y 2 - Portafolio de Inversiones y Selección de Personal

In [1]:
import random

class AlgoritmoGenetico:
    """
    Clase genérica y reutilizable para implementar un Algoritmo Genético (AG)
    con genotipo binario, que puede resolver distintos problemas de optimización
    combinatoria (Mochila, Portafolio, Selección de Personal, etc.).
    """

    def __init__(self, population_size, chromosome_length, pc, pm,
                 fitness_func, decode_func, selection_method='tournament',
                 tournament_size=3, elitism=True):
        self.population_size = population_size
        self.chromosome_length = chromosome_length
        self.pc = pc                      # Probabilidad de cruzamiento
        self.pm = pm                      # Probabilidad de mutación
        self.fitness_func = fitness_func  # Recibe un FENOTIPO (ya decodificado)
        self.decode_func = decode_func    # Convierte genotipo -> fenotipo
        self.selection_method = selection_method
        self.tournament_size = tournament_size
        self.elitism = elitism

        self.population = [self._generar_individuo() for _ in range(population_size)]
        self.history = []  # Guarda el mejor fitness de cada generación

    def _generar_individuo(self):
        return [random.randint(0, 1) for _ in range(self.chromosome_length)]

    def _evaluar_poblacion(self):
        """Decodifica y calcula el fitness de cada individuo de la población."""
        fitnesses = []
        for genotype in self.population:
            phenotype = self.decode_func(genotype)
            fitness = self.fitness_func(phenotype)
            fitnesses.append(fitness)
        return fitnesses

    def _seleccion_torneo(self, fitnesses):
        participantes = random.sample(range(len(self.population)), self.tournament_size)
        mejor_idx = max(participantes, key=lambda i: fitnesses[i])
        return self.population[mejor_idx][:]

    def _seleccion_ruleta(self, fitnesses):
        minimo = min(fitnesses)
        ajustados = [f - minimo + 1e-6 for f in fitnesses]
        total = sum(ajustados)
        pick = random.uniform(0, total)
        acumulado = 0
        for i, f in enumerate(ajustados):
            acumulado += f
            if acumulado >= pick:
                return self.population[i][:]
        return self.population[-1][:]

    def _seleccionar(self, fitnesses):
        if self.selection_method == 'tournament':
            return self._seleccion_torneo(fitnesses)
        elif self.selection_method == 'roulette':
            return self._seleccion_ruleta(fitnesses)
        raise ValueError(f"Método de selección no soportado: {self.selection_method}")

    def _cruzar(self, padre1, padre2):
        """Cruzamiento de un solo punto."""
        if random.random() > self.pc:
            return padre1[:], padre2[:]
        punto = random.randint(1, self.chromosome_length - 1)
        hijo1 = padre1[:punto] + padre2[punto:]
        hijo2 = padre2[:punto] + padre1[punto:]
        return hijo1, hijo2

    def _mutar(self, cromosoma):
        return [1 - gen if random.random() < self.pm else gen for gen in cromosoma]

    def run(self, num_generations):
        best_genotype_global = None
        best_fitness_global = -float('inf')
        self.history = []

        for gen in range(num_generations):
            fitnesses = self._evaluar_poblacion()
            idx_mejor = fitnesses.index(max(fitnesses))

            if fitnesses[idx_mejor] > best_fitness_global:
                best_fitness_global = fitnesses[idx_mejor]
                best_genotype_global = self.population[idx_mejor][:]
            self.history.append(max(fitnesses))

            nueva_poblacion = []
            if self.elitism:
                nueva_poblacion.append(self.population[idx_mejor][:])

            while len(nueva_poblacion) < self.population_size:
                padre1 = self._seleccionar(fitnesses)
                padre2 = self._seleccionar(fitnesses)
                hijo1, hijo2 = self._cruzar(padre1, padre2)
                hijo1 = self._mutar(hijo1)
                hijo2 = self._mutar(hijo2)
                nueva_poblacion.append(hijo1)
                if len(nueva_poblacion) < self.population_size:
                    nueva_poblacion.append(hijo2)

            self.population = nueva_poblacion

        return best_genotype_global, best_fitness_global

In [2]:
# ---------- Ejemplo del Módulo 3: Problema de la Mochila ----------
random.seed(42)

ITEMS = [
    {'name': 'Libro', 'weight': 2, 'value': 10},
    {'name': 'Laptop', 'weight': 5, 'value': 100},
    {'name': 'Agua', 'weight': 1, 'value': 5},
    {'name': 'Snacks', 'weight': 3, 'value': 20},
    {'name': 'Linterna', 'weight': 1, 'value': 15},
    {'name': 'Cámara', 'weight': 4, 'value': 80},
    {'name': 'Tienda', 'weight': 10, 'value': 60},
    {'name': 'Botiquín', 'weight': 2, 'value': 25},
]
MAX_CAPACITY = 15
PENALTY_FACTOR = 5

def decode_knapsack(genotype):
    """Genotipo -> Fenotipo: (peso_total, valor_total, ítems seleccionados)."""
    total_weight = 0
    total_value = 0
    selected_items_indices = []
    for i, bit in enumerate(genotype):
        if bit == 1:
            total_weight += ITEMS[i]['weight']
            total_value += ITEMS[i]['value']
            selected_items_indices.append(i)
    return {'total_weight': total_weight, 'total_value': total_value,
            'selected_items_indices': selected_items_indices}

def fitness_knapsack(phenotype):
    """Valor total, penalizado si se excede la capacidad máxima."""
    total_weight = phenotype['total_weight']
    total_value = phenotype['total_value']
    if total_weight > MAX_CAPACITY:
        return total_value - PENALTY_FACTOR * (total_weight - MAX_CAPACITY)
    return float(total_value)

print("--- Ejecutando AG para el Problema de la Mochila ---")
knapsack_ga = AlgoritmoGenetico(
    population_size=50, chromosome_length=len(ITEMS), pc=0.8, pm=0.01,
    fitness_func=fitness_knapsack, decode_func=decode_knapsack,
    selection_method='tournament', tournament_size=5, elitism=True,
)
best_genotype, best_fitness = knapsack_ga.run(100)
best_phenotype = decode_knapsack(best_genotype)
selected_items = [ITEMS[i]['name'] for i in best_phenotype['selected_items_indices']]

print(f"Mejor Genotipo: {''.join(map(str, best_genotype))}")
print(f"Mejor Fitness (Valor Total): {best_fitness:.2f}")
print(f"Peso Total: {best_phenotype['total_weight']} / Capacidad Máxima: {MAX_CAPACITY}")
print(f"Ítems Seleccionados: {', '.join(selected_items)}")

--- Ejecutando AG para el Problema de la Mochila ---
Mejor Genotipo: 01011111
Mejor Fitness (Valor Total): 250.00
Peso Total: 25 / Capacidad Máxima: 15
Ítems Seleccionados: Laptop, Snacks, Linterna, Cámara, Tienda, Botiquín


In [3]:
# ---------- Ejercicio 1: Portafolio de Inversiones ----------
random.seed(42)

PROYECTOS = [
    {'name': 'Proyecto 1',  'costo': 20, 'retorno': 65},
    {'name': 'Proyecto 2',  'costo': 30, 'retorno': 80},
    {'name': 'Proyecto 3',  'costo': 15, 'retorno': 30},
    {'name': 'Proyecto 4',  'costo': 35, 'retorno': 90},
    {'name': 'Proyecto 5',  'costo': 10, 'retorno': 22},
    {'name': 'Proyecto 6',  'costo': 25, 'retorno': 55},
    {'name': 'Proyecto 7',  'costo': 40, 'retorno': 105},
    {'name': 'Proyecto 8',  'costo': 18, 'retorno': 40},
    {'name': 'Proyecto 9',  'costo': 22, 'retorno': 48},
    {'name': 'Proyecto 10', 'costo': 12, 'retorno': 28},
]
PRESUPUESTO_MAXIMO = 100
FACTOR_PENALIZACION_PORTAFOLIO = 20  # fuerte, a propósito (ver Paso 9)

def decode_portfolio(genotype):
    """Genotipo -> Fenotipo: (costo_total, retorno_total, proyectos elegidos)."""
    costo_total = 0
    retorno_total = 0
    indices_seleccionados = []
    for i, bit in enumerate(genotype):
        if bit == 1:
            costo_total += PROYECTOS[i]['costo']
            retorno_total += PROYECTOS[i]['retorno']
            indices_seleccionados.append(i)
    return {'costo_total': costo_total, 'retorno_total': retorno_total,
            'indices_seleccionados': indices_seleccionados}

def fitness_portfolio(phenotype):
    """Retorno total, penalizado fuertemente si se excede el presupuesto."""
    costo_total = phenotype['costo_total']
    retorno_total = phenotype['retorno_total']
    if costo_total > PRESUPUESTO_MAXIMO:
        exceso = costo_total - PRESUPUESTO_MAXIMO
        return retorno_total - FACTOR_PENALIZACION_PORTAFOLIO * exceso
    return float(retorno_total)

print("--- Ejecutando AG para el Portafolio de Inversiones ---")
portfolio_ga = AlgoritmoGenetico(
    population_size=50, chromosome_length=len(PROYECTOS), pc=0.8, pm=0.05,
    fitness_func=fitness_portfolio, decode_func=decode_portfolio,
    selection_method='tournament', tournament_size=3, elitism=True,
)
best_genotype, best_fitness = portfolio_ga.run(80)
best_phenotype = decode_portfolio(best_genotype)
nombres_elegidos = [PROYECTOS[i]['name'] for i in best_phenotype['indices_seleccionados']]

print(f"Mejor Genotipo: {''.join(map(str, best_genotype))}")
print(f"Proyectos Seleccionados: {', '.join(nombres_elegidos)}")
print(f"Costo Total: {best_phenotype['costo_total']} / Presupuesto: {PRESUPUESTO_MAXIMO}")
print(f"Retorno Total (Fitness): {best_phenotype['retorno_total']}")

--- Ejecutando AG para el Portafolio de Inversiones ---
Mejor Genotipo: 1100101000
Proyectos Seleccionados: Proyecto 1, Proyecto 2, Proyecto 5, Proyecto 7
Costo Total: 100 / Presupuesto: 100
Retorno Total (Fitness): 272


In [4]:
# ---------- Ejercicio 2: Selección de Personal Estricta ----------
random.seed(42)

CANDIDATOS = [
    {'name': f'Candidato {i+1}', 'habilidad': h}
    for i, h in enumerate([78, 65, 92, 55, 88, 70, 60, 95, 82, 45, 73, 67])
]
TAMANO_EQUIPO_REQUERIDO = 5
FACTOR_PENALIZACION_EQUIPO = 100  # tiene que ser alto: ver nota abajo

def decode_team(genotype):
    """Genotipo -> Fenotipo: (n_seleccionados, habilidad_total, candidatos elegidos)."""
    seleccionados_idx = [i for i, bit in enumerate(genotype) if bit == 1]
    habilidad_total = sum(CANDIDATOS[i]['habilidad'] for i in seleccionados_idx)
    return {'n_seleccionados': len(seleccionados_idx),
            'habilidad_total': habilidad_total,
            'seleccionados_idx': seleccionados_idx}

def fitness_team(phenotype):
    """Habilidad total, penalizada si el equipo NO tiene exactamente 5 integrantes."""
    diferencia = abs(phenotype['n_seleccionados'] - TAMANO_EQUIPO_REQUERIDO)
    penalizacion = diferencia * FACTOR_PENALIZACION_EQUIPO
    return phenotype['habilidad_total'] - penalizacion

print("--- Ejecutando AG para la Selección de Personal ---")
team_ga = AlgoritmoGenetico(
    population_size=50, chromosome_length=len(CANDIDATOS), pc=0.8, pm=0.05,
    fitness_func=fitness_team, decode_func=decode_team,
    selection_method='tournament', tournament_size=3, elitism=True,
)
best_genotype2, best_fitness2 = team_ga.run(80)
best_phenotype2 = decode_team(best_genotype2)
nombres_equipo = [CANDIDATOS[i]['name'] for i in best_phenotype2['seleccionados_idx']]

print(f"Mejor Genotipo: {''.join(map(str, best_genotype2))}")
print(f"Equipo Seleccionado: {', '.join(nombres_equipo)}")
print(f"Tamaño del equipo: {best_phenotype2['n_seleccionados']} (requerido: {TAMANO_EQUIPO_REQUERIDO})")
print(f"Habilidad Técnica Total: {best_phenotype2['habilidad_total']}")

--- Ejecutando AG para la Selección de Personal ---
Mejor Genotipo: 101010011000
Equipo Seleccionado: Candidato 1, Candidato 3, Candidato 5, Candidato 8, Candidato 9
Tamaño del equipo: 5 (requerido: 5)
Habilidad Técnica Total: 435
